# Station-Buffered, Elevation-Stratified Pixel Sampling
### Lake Neusiedl Reed Belt (Austrian Side) — Figure 4.2

Python port of the GEE JavaScript sampling script, extended to produce a publication-ready
static map (station labels, elevation legend, scale bar, north arrow) instead of a GEE Map
panel screenshot.

**Before running:** `pip install earthengine-api geemap geopandas shapely rasterio matplotlib-scalebar --break-system-packages`


In [ ]:
# ============================================================
# CELL 0 — SETUP
# ============================================================

import ee
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib_scalebar.scalebar import ScaleBar
import rasterio
from rasterio.plot import show as rio_show
import geemap
from shapely.geometry import shape, Point

try:
    ee.Initialize(project='YOUR_GEE_PROJECT')
    print("Earth Engine initialized successfully")
except Exception:
    ee.Authenticate()
    ee.Initialize(project='YOUR_GEE_PROJECT')


In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

ASSET_PATH = "projects/YOUR_GEE_PROJECT/assets/DGM_wetland_AT"
ELEV_MIN   = 114.5
ELEV_MAX   = 117.0
BIN_SIZE   = 0.1
SCALE      = 30
SEED       = 42
TARGET_PER_STATION      = 100
PX_PER_STRAT_INITIAL    = 6

OUT_DIR = "<PROJECT_ROOT>/result/writting/4.3_sampling/"
os.makedirs(OUT_DIR, exist_ok=True)

# Podersdorf was excluded (excessive spatial clustering, insufficient elevation-bin
# spread within buffer) BEFORE coordinates were finalized for this script, so they
# are not available here. Fill these in and set SHOW_PODERSDORF_MARKER = True if you
# want its location marked on the map for context.
SHOW_PODERSDORF_MARKER = False
PODERSDORF_LON = None   # e.g. 16.8148
PODERSDORF_LAT = None   # e.g. 47.8508

print("Configuration loaded. Output directory:", OUT_DIR)


In [ ]:
# ============================================================
# CELL 2 — STATIONS + BUFFERS (Podersdorf excluded)
# ============================================================

stations_wgs = [
    {"station": "Breitenbrunn", "lon": 16.770256,  "lat": 47.916203, "buffer_m": 2500},
    {"station": "Moerbisch",    "lon": 16.698516,  "lat": 47.751756, "buffer_m": 2000},
    {"station": "Neusiedl",     "lon": 16.835600,  "lat": 47.929900, "buffer_m": 2500},
    {"station": "Rust",         "lon": 16.695700,  "lat": 47.801700, "buffer_m": 2000},
    {"station": "Illmitz",      "lon": 16.752600,  "lat": 47.770000, "buffer_m": 2500},
    {"station": "Apetlon",      "lon": 16.7490326, "lat": 47.6814064, "buffer_m": 4000},
]

# Distinct, colorblind-friendly color per station, reused for buffer outline,
# sample points, and legend so a reader can trace one station through the figure.
STATION_COLORS = {
    "Breitenbrunn": "#e07b39",
    "Moerbisch":    "#01696f",
    "Neusiedl":     "#8856a7",
    "Rust":         "#d62728",
    "Illmitz":      "#2c7fb8",
    "Apetlon":      "#5a3d2b",
}

print("Stations configured:", [s["station"] for s in stations_wgs])


In [ ]:
# ============================================================
# CELL 3 — LOAD DEM, MASK TO ELEVATION RANGE, COMPUTE BINS
# ============================================================

dem = ee.Image(ASSET_PATH).rename('elevation')

elev_mask  = dem.gte(ELEV_MIN).And(dem.lte(ELEV_MAX))
dem_masked = dem.updateMask(elev_mask)

elev_binned = (dem_masked.subtract(ELEV_MIN)
               .divide(BIN_SIZE)
               .floor()
               .rename('elev_bin')
               .toInt())

dem_with_bin = elev_binned.addBands(dem_masked)

print("DEM loaded and masked to", ELEV_MIN, "-", ELEV_MAX, "m a.s.l.")


In [ ]:
# ============================================================
# CELL 4 — PER-STATION BUFFERED STRATIFIED SAMPLING
# ------------------------------------------------------------
# Same two-pass fallback logic as the original JS script:
# stratified draw, then oversample-and-limit fill passes at 4x
# and 8x the residual deficit if the stratified draw falls short.
# ============================================================

def sample_station(st):
    pt = ee.Geometry.Point([st['lon'], st['lat']])
    buffer = pt.buffer(st['buffer_m'])
    dem_station = dem_with_bin.clip(buffer)

    strat_samples = dem_station.stratifiedSample(
        numPoints=PX_PER_STRAT_INITIAL,
        classBand='elev_bin',
        region=buffer,
        scale=SCALE,
        seed=SEED,
        geometries=True,
        dropNulls=True
    )

    combined  = strat_samples
    n_current = combined.size()

    deficit1 = ee.Number(TARGET_PER_STATION).subtract(n_current).max(0)
    fill1 = ee.FeatureCollection(ee.Algorithms.If(
        deficit1.gt(0),
        dem_station.select(['elevation', 'elev_bin']).sample(
            region=buffer, scale=SCALE,
            numPixels=deficit1.multiply(4).max(50),
            seed=SEED + 1, geometries=True, dropNulls=True
        ).limit(deficit1),
        ee.FeatureCollection([])
    ))
    combined  = combined.merge(fill1)
    n_current = combined.size()

    deficit2 = ee.Number(TARGET_PER_STATION).subtract(n_current).max(0)
    fill2 = ee.FeatureCollection(ee.Algorithms.If(
        deficit2.gt(0),
        dem_station.select(['elevation', 'elev_bin']).sample(
            region=buffer, scale=SCALE,
            numPixels=deficit2.multiply(8).max(80),
            seed=SEED + 2, geometries=True, dropNulls=True
        ).limit(deficit2),
        ee.FeatureCollection([])
    ))
    combined = combined.merge(fill2)
    combined = ee.FeatureCollection(combined.toList(TARGET_PER_STATION))

    def tag(f):
        bin_ = f.getNumber('elev_bin')
        dist_km = f.geometry().distance(pt).divide(1000)
        return f.set({
            'station':               st['station'],
            'station_lon':           st['lon'],
            'station_lat':           st['lat'],
            'buffer_m':              st['buffer_m'],
            'distance_to_station_km': dist_km,
            'bin_lower_m':           ee.Number(ELEV_MIN).add(bin_.multiply(BIN_SIZE)),
            'bin_upper_m':           ee.Number(ELEV_MIN).add(bin_.add(1).multiply(BIN_SIZE)),
            'pixel_id':              ee.String(st['station']).cat('_').cat(f.id())
        })

    combined = combined.map(tag)
    return combined, buffer


In [ ]:
# ============================================================
# CELL 5 — RUN SAMPLING FOR ALL STATIONS
# ============================================================

all_samples = ee.FeatureCollection([])
station_buffers = {}

for st in stations_wgs:
    combined, buffer_geom = sample_station(st)
    all_samples = all_samples.merge(combined)
    station_buffers[st['station']] = buffer_geom
    print(f"{st['station']} — pixels sampled: {combined.size().getInfo()}")

print("\nTOTAL pixels sampled (all stations):", all_samples.size().getInfo())


In [ ]:
# ============================================================
# CELL 6 — EXPORT SAMPLED PIXELS TO LOCAL CSV
# ============================================================

features = all_samples.getInfo()['features']

rows = []
for f in features:
    props = f['properties']
    coords = f['geometry']['coordinates']
    props['lon'] = coords[0]
    props['lat'] = coords[1]
    rows.append(props)

df = pd.DataFrame(rows)
cols = ['pixel_id', 'station', 'station_lon', 'station_lat', 'buffer_m',
        'distance_to_station_km', 'elev_bin', 'bin_lower_m', 'bin_upper_m',
        'elevation', 'lon', 'lat']
df = df[[c for c in cols if c in df.columns]]

csv_path = os.path.join(OUT_DIR, 'FLATS_station_buffered_pixels_6stations.csv')
df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} rows to {csv_path}")


In [ ]:
# ============================================================
# CELL 7 — EXPORT DEM RASTER (LOCAL GEOTIFF, EXPLICIT PROJECTED CRS)
# ------------------------------------------------------------
# FIX: the previous version did not set an explicit output CRS,
# so the exported raster likely came back in a different
# coordinate system than the lon/lat buffers and points used for
# overlay — the two layers would not have aligned. Everything is
# now forced into UTM zone 33N (EPSG:32633, meters), which covers
# this part of Austria and gives a properly proportioned map.
# ============================================================

MAP_CRS = 'EPSG:32633'  # UTM zone 33N — correct zone for Lake Neusiedl (~16.7-16.9°E)

all_buffers_fc = ee.FeatureCollection(
    [ee.Feature(b) for b in station_buffers.values()]
)
region = all_buffers_fc.geometry().bounds().buffer(500).bounds()

dem_tif_path = os.path.join(OUT_DIR, 'dem_masked_for_map.tif')
geemap.ee_export_image(
    dem_masked, filename=dem_tif_path, scale=SCALE,
    crs=MAP_CRS, region=region, file_per_band=False
)

# Diagnostic check — confirms the export actually worked before we try to plot it
with rasterio.open(dem_tif_path) as src:
    arr = src.read(1)
    valid = arr[arr != src.nodata] if src.nodata is not None else arr[~np.isnan(arr)]
    print("Raster CRS:", src.crs)
    print("Raster shape:", arr.shape)
    print("Raster bounds:", src.bounds)
    print("Valid (non-nodata) pixel count:", valid.size, "/", arr.size)
    if valid.size > 0:
        print("Valid pixel value range:", float(valid.min()), "-", float(valid.max()))
    else:
        print("WARNING: raster contains no valid data — check ASSET_PATH and region.")


In [ ]:
# ============================================================
# CELL 8 — BUILD THE PUBLICATION-READY MAP (Figure 4.2)
# ------------------------------------------------------------
# Raster, buffers, and points are all reprojected to the same
# projected CRS (MAP_CRS) before plotting, so they are guaranteed
# to align. Aspect ratio is trivially correct in a projected CRS
# (meters on both axes), so no latitude-dependent correction is
# needed here, unlike the previous lon/lat version.
# ============================================================

# --- load raster, mask nodata for clean display ---
with rasterio.open(dem_tif_path) as src:
    dem_arr = src.read(1).astype(float)
    if src.nodata is not None:
        dem_arr[dem_arr == src.nodata] = np.nan
    raster_bounds = src.bounds
    raster_crs = src.crs

extent = (raster_bounds.left, raster_bounds.right, raster_bounds.bottom, raster_bounds.top)

# --- buffers: EE geometry -> shapely -> GeoDataFrame -> reproject ---
buffer_records = []
for name, geom in station_buffers.items():
    buffer_records.append({'station': name, 'geometry': shape(geom.getInfo())})
buffers_gdf = gpd.GeoDataFrame(buffer_records, crs='EPSG:4326').to_crs(MAP_CRS)

# --- sample points: DataFrame lon/lat -> GeoDataFrame -> reproject ---
points_gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs='EPSG:4326'
).to_crs(MAP_CRS)

# --- station center points, reprojected, for label placement ---
centers = gpd.GeoDataFrame(
    stations_wgs,
    geometry=[Point(s['lon'], s['lat']) for s in stations_wgs],
    crs='EPSG:4326'
).to_crs(MAP_CRS)

fig, ax = plt.subplots(figsize=(9, 12))

# --- background: elevation raster ---
im = ax.imshow(dem_arr, cmap='YlOrBr_r', vmin=ELEV_MIN, vmax=ELEV_MAX,
                extent=extent, origin='upper', zorder=1)
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label('Elevation, ALS DEM masked to wetland boundary (m a.s.l.)', fontsize=9)

# --- station buffers + labels ---
for _, row in buffers_gdf.iterrows():
    name = row['station']
    color = STATION_COLORS[name]
    xs, ys = row.geometry.exterior.xy
    ax.plot(xs, ys, color=color, linewidth=1.8, zorder=3)
    center = centers.loc[centers['station'] == name, 'geometry'].values[0]
    label_y = max(ys)
    ax.annotate(name, xy=(center.x, label_y), xytext=(0, 6),
                textcoords='offset points', ha='center', fontsize=10,
                fontweight='bold', color=color, zorder=5)

# --- sampled pixels, colored per station ---
for name in STATION_COLORS:
    sub = points_gdf[points_gdf['station'] == name]
    ax.scatter(sub.geometry.x, sub.geometry.y, s=14, color=STATION_COLORS[name],
               edgecolor='white', linewidth=0.4, zorder=4)

# --- optional Podersdorf marker ---
if SHOW_PODERSDORF_MARKER and PODERSDORF_LON is not None:
    pod_pt = gpd.GeoSeries([Point(PODERSDORF_LON, PODERSDORF_LAT)], crs='EPSG:4326').to_crs(MAP_CRS).iloc[0]
    ax.scatter([pod_pt.x], [pod_pt.y], marker='x', s=90, color='black', linewidth=2, zorder=6)
    ax.annotate('Podersdorf (excluded)', xy=(pod_pt.x, pod_pt.y),
                xytext=(8, 8), textcoords='offset points', fontsize=9,
                style='italic', color='black', zorder=6)

ax.set_aspect('equal')  # correct by construction in a projected (meters) CRS

# --- scale bar + north arrow (trivial in meters, no lat-dependent conversion needed) ---
ax.add_artist(ScaleBar(dx=1, units='m', location='lower left', box_alpha=0.7))
ax.annotate('N', xy=(0.96, 0.90), xytext=(0.96, 0.82),
            xycoords='axes fraction', ha='center', fontsize=13, fontweight='bold',
            arrowprops=dict(facecolor='black', width=3, headwidth=10, shrink=0.05))

# --- legend ---
legend_elements = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='grey',
           markeredgecolor='white', markersize=7, label='Sampled pixel'),
    Line2D([0], [0], color='grey', linewidth=1.8, label='Station buffer boundary'),
]
if SHOW_PODERSDORF_MARKER and PODERSDORF_LON is not None:
    legend_elements.append(
        Line2D([0], [0], marker='x', color='black', linewidth=0,
               markersize=9, label='Excluded station (Podersdorf)')
    )
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.85)

ax.set_title('Station-Buffered Elevation Sample\nAustrian Reed Belt, Lake Neusiedl',
              fontsize=13, fontweight='bold')
ax.set_xlabel(f'Easting ({MAP_CRS}, m)'); ax.set_ylabel(f'Northing ({MAP_CRS}, m)')

fig.text(0.5, 0.005,
          'Pixels sampled within a buffer around each of 6 gauge stations '
          '(Podersdorf excluded — insufficient spatial spread),\n'
          'stratified by 10 cm elevation bins, target = 100 pixels per station. '
          'DEM source: Land Burgenland ALS 2019.',
          ha='center', fontsize=8, color='#555555')

plt.tight_layout(rect=[0, 0.03, 1, 1])
fig_path = os.path.join(OUT_DIR, 'figure4_2_station_buffered_sample_map.png')
plt.savefig(fig_path, dpi=320, bbox_inches='tight')
plt.show()
print("Figure saved to", fig_path)
